# Custom SVM (Classification & Regression)

In [7]:
import numpy as np

In [8]:
class CustomSVM:
    def __init__(self, mode='classification', C=1.0, tol=1e-3, eps=0.1, lr=0.001, epochs=500):
        self.mode = mode
        self.C = C
        self.tol = tol
        self.eps = eps
        self.lr = lr
        self.epochs = epochs
        self.models = {}
        self.w = None
        self.b = None

    def _compute_e(self, X, y, alpha, b, X_train, y_train, i):
        prediction = np.dot(np.dot(alpha * y_train, X_train), X[i].T) + b
        return prediction - y[i]

    def _fit_binary_smo(self, X, y):
        n_samples, n_features = X.shape
        alpha = np.zeros(n_samples)
        b = 0.0
        E = np.zeros(n_samples)
        for i in range(n_samples):
            E[i] = self._compute_e(X, y, alpha, b, X, y, i)

        for epoch in range(self.epochs):
            num_changed_alphas = 0
            for i in range(n_samples):
                Ei = E[i]
                viol = y[i] * Ei
                if (viol < -self.tol and alpha[i] < self.C) or (viol > self.tol and alpha[i] > 0):
                    Ej_all = np.abs(E - Ei)
                    Ej_all[i] = -1
                    j = int(np.argmax(Ej_all))
                    Ej = E[j]
                    alpha_i_old, alpha_j_old = alpha[i].copy(), alpha[j].copy()
                    if y[i] != y[j]:
                        L = max(0.0, alpha[j] - alpha[i])
                        H = min(self.C, self.C + alpha[j] - alpha[i])
                    else:
                        L = max(0.0, alpha[i] + alpha[j] - self.C)
                        H = min(self.C, alpha[i] + alpha[j])
                    if L == H:
                        continue
                    eta = 2.0 * np.dot(X[i], X[j]) - np.dot(X[i], X[i]) - np.dot(X[j], X[j])
                    if eta >= 0:
                        continue
                    alpha[j] -= (y[j] * (Ei - Ej)) / eta
                    alpha[j] = np.clip(alpha[j], L, H)
                    if np.abs(alpha[j] - alpha_j_old) < 1e-5:
                        continue
                    alpha[i] += y[i] * y[j] * (alpha_j_old - alpha[j])
                    b1 = b - Ei - y[i] * (alpha[i] - alpha_i_old) * np.dot(X[i], X[i]) - y[j] * (alpha[j] - alpha_j_old) * np.dot(X[i], X[j])
                    b2 = b - Ej - y[i] * (alpha[i] - alpha_i_old) * np.dot(X[i], X[j]) - y[j] * (alpha[j] - alpha_j_old) * np.dot(X[j], X[j])
                    if 0 < alpha[i] < self.C:
                        b = b1
                    elif 0 < alpha[j] < self.C:
                        b = b2
                    else:
                        b = (b1 + b2) / 2.0
                    for k in range(n_samples):
                        E[k] = self._compute_e(X, y, alpha, b, X, y, k)
                    num_changed_alphas += 1
            if num_changed_alphas == 0:
                break
        support_indices = alpha > 1e-5
        return alpha[support_indices], b, X[support_indices], y[support_indices]

    def _fit_regression_gd(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0
        for _ in range(self.epochs):
            predictions = np.dot(X, self.w) + self.b
            errors = predictions - y
            dw = self.w.copy()
            db = 0.0
            upper_mask = errors > self.eps
            if np.any(upper_mask):
                dw += self.C * np.dot(X[upper_mask].T, np.ones(np.sum(upper_mask)))
                db += self.C * np.sum(upper_mask)
            lower_mask = errors < -self.eps
            if np.any(lower_mask):
                dw -= self.C * np.dot(X[lower_mask].T, np.ones(np.sum(lower_mask)))
                db -= self.C * np.sum(lower_mask)
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def fit(self, X, y):
        if self.mode == 'regression':
            self._fit_regression_gd(X, y)
        elif self.mode == 'classification':
            self.classes_ = np.unique(y)
            for c in self.classes_:
                y_binary = np.where(y == c, 1, -1)
                alpha, b, support_X, support_y = self._fit_binary_smo(X, y_binary)
                self.models[c] = (alpha, b, support_X, support_y)
        return self

    def predict(self, X):
        if self.mode == 'regression':
            return np.dot(X, self.w) + self.b
        elif self.mode == 'classification':
            scores = np.zeros((X.shape[0], len(self.classes_)))
            for idx, c in enumerate(self.classes_):
                alpha, b, support_X, support_y = self.models[c]
                if len(alpha) > 0:
                    scores[:, idx] = np.dot(np.dot(alpha * support_y, support_X), X.T) + b
                else:
                    scores[:, idx] = b
            return self.classes_[np.argmax(scores, axis=1)]

# Model Testing

In [9]:
from sklearn.datasets import make_blobs, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error

X_cls, y_cls = make_blobs(n_samples=100, n_features=2, centers=3, random_state=42)
clf = CustomSVM(mode='classification', C=1.0, epochs=200)
clf.fit(X_cls, y_cls)
preds_cls = clf.predict(X_cls)
print(f"Classification Accuracy: {accuracy_score(y_cls, preds_cls):.2f}")

X_reg, y_reg = make_regression(n_samples=100, n_features=2, noise=10.0, random_state=42)
reg = CustomSVM(mode='regression', C=10.0, eps=1.0, lr=0.001, epochs=1000)
reg.fit(X_reg, y_reg)
preds_reg = reg.predict(X_reg)
print(f"Regression MSE: {mean_squared_error(y_reg, preds_reg):.2f}")

Classification Accuracy: 1.00
Regression MSE: 117.72
